# Fixed 16 MW data center: HPP LCoE and reliability

Original Parallel EGO sizes wind, PV, battery power and duration. The hourly EMS and economics are unchanged. The objective is annualized HPP cost divided by delivered **facility** energy, including partial supply. Reliability remains a constraint.

Each run creates a new timestamped experiment under `hpp_sizing_runs/lcoe_reliability_frontier`. Native checkpoints, logs and hourly results are retained inside each target folder. Time-limited results are validated incumbents, not global-optimality certificates.


In [ ]:
import os
os.environ['OPENMDAO_REPORTS'] = '0'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
from pathlib import Path
import matplotlib.pyplot as plt
import hydesign
from hydesign.sizing_native_hpp_sizing_base import (
    read_configuration, run_lcoe_reliability_experiment,
    load_lcoe_reliability_experiment,
)
repo = Path(hydesign.__file__).resolve().parent.parent
config_path = repo / 'hydesign/examples/thesis_simple_datacenter/simple_datacenter.yml'


## Configuration
Change targets and the per-target search budget here. For a pilot, use `[0.70, 0.90]` and a small budget. To inspect an existing experiment without running EGO, set `saved_experiment` to its folder. Regenerated exports go into a new `reports_...` folder, preserving earlier files.


In [ ]:
reliability_targets = [0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.97, 0.99, 0.999, 1.00]
search_budget_seconds = 600  # Per target; final independent validation adds time.
output_dir = repo / 'hpp_sizing_runs/lcoe_reliability_frontier'
saved_experiment = None  # Or Path(r'C:/.../experiment_YYYYMMDD_HHMMSSZ_...')
regenerate_saved_figures = False  # True exports new PNG/PDF/CSV reports when loading.

config = read_configuration(config_path)
config.update(facility_load_mw=16, pue=1.15,
              hpp_sizing_objective='minimize_lcoe',
              battery_power_fluctuation_penalty=0.0001,
              hpp_sizing_time_limit_seconds=search_budget_seconds)
for name in ('Nwt', 'solar_MW [MW AC]', 'b_P [MW]', 'b_E_h [h]'):
    config['variables'][name]['var_type'] = 'design'
# Existing YAML bounds and economics are retained. Optional search overrides:
# config.update(n_doe=16, n_procs=2, n_clusters=2, max_iter=100,
#               min_conv_iter=100, npred=2000, n_seed=0, ems_threads=1)
print('Targets:', reliability_targets, '| Search seconds per target:', search_budget_seconds)


## Run or load
The summary is saved after every completed target, including failure statuses. Pending targets remain in the table after interruption. A fresh optimization always uses a new experiment folder.


In [ ]:
if saved_experiment is None:
    experiment_dir = run_lcoe_reliability_experiment(
        config_path, config, reliability_targets, output_dir)
else:
    experiment_dir = Path(saved_experiment)
print('Experiment:', experiment_dir)


## Results
Points use achieved hourly reliability. The dashed line joins only nondominated selected designs and represents an approximate observed frontier. Infeasible, unresolved and pending cases remain in the saved table but are not plotted. The complete evaluated-design history is in each target's `evaluations.csv`.


In [ ]:
summary, figure = load_lcoe_reliability_experiment(
    experiment_dir, save_figures=bool(saved_experiment is not None and regenerate_saved_figures))
display(summary)
plt.show()
